METODE EMBEDDING
Doc2Vec(Paragraf Vector): Pengembangan dari word2vec untuk mempresentasikan seluruh documen atau paragraf menjadi vektor

Vektor dalam kode Doc2Vec adalah representasi teks dalam bentuk angka. Setiap dokumen atau paragraf diubah menjadi deretan angka (misalnya 100 angka) agar bisa dipahami dan diproses oleh komputer. Vektor ini menyimpan informasi makna dari teks, sehingga dokumen yang mirip akan memiliki vektor yang juga mirip. Dengan vektor inilah model bisa membandingkan, mencari kemiripan, atau menganalisis teks secara matematis.

In [9]:
import pandas as pd
import numpy as np
from google.colab import drive

fungsi code dibawah untuk memanggil dataset yang tersimpan atau yang sudah di upload  di google drive

In [16]:
drive.mount('/content/drive')
csv_path = '/content/drive/My Drive/nlp/NEWS.xlsx'
df = pd.read_excel(csv_path)
print(df.head(10))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
                                               title  \
0  Depo Plumpang Terbakar, Anggota DPR Minta Pert...   
1  Jokowi Perintahkan Wapres Ma'ruf Amin Tinjau L...   
2  HNW Mendukung Jamaah Umroh First Travel Dapatk...   
3  Tim Dokkes Polri Telah Terima 14 Kantong Jenaz...   
4  Bamsoet Ajak Komunitas Otomotif Kembangkan Per...   
5  Korban Tewas Kebakaran Depo Plumpang 17 Orang,...   
6  14 Tahun Berkarya, PT SMI Siap Bangun Indonesi...   
7  Penundaan Pemilu 2024 Bisa Buat Jokowi 3 Perio...   
8  Mabes Polri Selidiki Penyebab Kebakaran Depo P...   
9  Jokowi Ingin Pindahkan Depo Pertamina dari Pem...   

                                             content  
0  TEMPO.CO, Jakarta - Anggota Komisi VII DPR RI ...  
1  TEMPO.CO, Jakarta - Presiden Joko Widodo atau ...  
2  INFO NASIONAL - Wakil Ketua MPR RI Dr. H. M. H...  
3  TEMPO.CO, Jakarta - Tim Kedokte

AMBIL KOLOM TEKS

Kode digunakan untuk mengambil kolom teks dari dataset dan mengubahnya menjadi list.

df['content'] → mengambil kolom content
astype(str) → memastikan semua data berupa teks
tolist() → mengubah menjadi list Python

In [18]:
data = df['content'].astype(str).tolist()

PROCESSING DAN TAGGING

word_tokenize(doc.lower()) → mengubah teks jadi huruf kecil lalu memecah menjadi kata-kata
enumerate(data) → memberi nomor (ID) tiap dokumen
TaggedDocument(...) → menggabungkan kata + ID dokumen
hasilnya adalah data yang sudah berupa kata-kata dan memiliki label (tag) sehingga bisa digunakan oleh model Doc2Vec.

In [22]:
!pip install gensim
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

from gensim.models.doc2vec import TaggedDocument
from nltk.tokenize import word_tokenize

tagged_data = [
    TaggedDocument(words=word_tokenize(doc.lower()), tags=[str(i)])
    for i, doc in enumerate(data)
]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Training Model Doc2Vec

In [24]:
from gensim.models import Doc2Vec

model = Doc2Vec(
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=50
)

model.build_vocab(tagged_data)
model.train(tagged_data, total_examples=model.corpus_count, epochs=model.epochs)

Ambil Vektor Dokumen
model.dv → kumpulan vektor semua dokumen
'0' → ID dokumen pertama
kode ini menampilkan representasi angka (vektor) dari dokumen pertama hasil training Doc2Vec.

In [25]:
print(model.dv['0'])  # dokumen pertama

[-0.6120788  -3.6741467  -0.8026751   1.3530084  -2.3138802  -0.25433603
  0.13301283 -0.64347607 -1.1685573   0.5335891  -0.99132377  2.1705415
 -2.705623   -1.0646061  -0.4128138  -0.2788839   0.86077183 -1.5277209
 -0.24159877 -0.32707423 -1.2372293   1.6087332  -0.8198007   0.03526971
  0.22349592 -0.15043378 -3.2895594  -0.7621351   0.5290113  -1.8499731
  2.7140043  -1.8662704   0.54538643 -1.0392666  -0.614052    0.02019916
  1.1513133  -0.41335613  0.0754225  -2.728224    0.9969586  -1.1492937
 -2.0477958   0.4154424  -1.5294915   0.46975672  0.61999196  0.05242099
  2.0763712  -1.419943    0.92540026  3.986254    0.37688783 -3.0626132
 -1.148312    1.1805798  -0.4262918   1.6817374  -0.9240178   1.3120248
  1.3053135   1.0318893  -0.3662845   0.03178167  0.4202541   0.04803549
 -1.4411471   0.71503824 -2.674459    3.06926    -0.81250507 -0.12959163
  0.10213839  0.8210505   3.5809128  -0.12619278  0.47482157  1.4473536
 -1.1614462   1.7250721   1.5596043   0.92838556 -1.613040

Uji Paragraf Baru

Kode ini digunakan untuk mengubah paragraf baru menjadi vektor.
test_doc → teks baru yang ingin diuji
word_tokenize(test_doc.lower()) → ubah ke huruf kecil lalu pecah jadi kata
infer_vector() → mengubah teks tersebut menjadi vektor berdasarkan model yang sudah dilatih
hasil print(vector) adalah representasi angka dari paragraf baru.

In [26]:
test_doc = "berita ekonomi indonesia"
vector = model.infer_vector(word_tokenize(test_doc.lower()))

print(vector)

[-0.05082923  0.11439274  0.16414869 -0.5317946   0.13075876 -0.22979221
  0.00752217  0.32140866 -0.06829171  0.12629291  0.1016921  -0.15664898
 -0.15955098 -0.2612406   0.45150176 -0.13120802 -0.25193158  0.16418119
 -0.49092063  0.04192736 -0.37039366 -0.29778272  0.6860575   0.3477247
 -0.09837019 -0.09535082 -0.16282362  0.1257824  -0.10110081  0.18990372
  0.3504383  -0.33229405  0.02367062 -0.09582315 -0.11540053 -0.09638988
  0.03861907 -0.26438162  0.34375355 -0.36240593  0.0510313   0.11381704
 -0.41484833 -0.07773355 -0.0054664  -0.00420641  0.17228682 -0.05122809
 -0.05329578 -0.14495838  0.28645274  0.10710205 -0.13869286 -0.2153436
 -0.24150738  0.22319563 -0.20648134  0.23097752 -0.0625113   0.30615258
  0.2656077  -0.22460052  0.12261805 -0.1352662   0.09930252  0.30523327
  0.4961738   0.19203863 -0.13705035  0.10836176  0.0629395  -0.28650525
 -0.2092783  -0.12889186  0.07937867  0.21483782 -0.10256211  0.03926988
 -0.29946265 -0.02715811  0.64639395 -0.3044265  -0.1

Cari Dokumen Paling Mirip

Kode ini digunakan untuk mencari dokumen yang paling mirip dengan teks yang sudah diubah menjadi vektor.

most_similar([vector], topn=5) → mencari 5 dokumen dengan kemiripan tertinggi
model.dv → kumpulan vektor semua dokumen

Hasil print(similar_docs) menampilkan ID dokumen dan nilai kemiripannya.

In [27]:
similar_docs = model.dv.most_similar([vector], topn=5)

print(similar_docs)

[('147', 0.4622330367565155), ('135', 0.4502106308937073), ('375', 0.44669315218925476), ('149', 0.4335330128669739), ('146', 0.42979374527931213)]
